# scoutfield 04 Evaluation Sweep

Full evaluation sweep, tau sensitivity, and figures.

Roadmap items 6 and 8. CPU is enough: evaluation is thousands of short episodes where per-step transfer latency dominates.

Run in bounded chunks so a killed session costs one chunk, not the whole sweep. To restart cleanly, delete BOTH results/_done.json and the results CSV — deleting one leaves the driver mixing old and new results with no warning.

> Logic lives in `scoutfield/` and `experiments/`. This notebook orchestrates: install, configure, call, display.


In [ ]:
# ---------------------------------------------------------------- bootstrap
# Requires: Internet ON, Accelerator GPU (None for the evaluation sweep),
# Persistence "Variables and Files".
#
# The repo is CLONED, not merely pip-installed. `pyproject.toml` ships the
# `scoutfield*` packages only, so `experiments/` and `configs/` would otherwise be
# missing and every run cell below would fail with "can't open file".
import os
import shutil
import subprocess
import sys

REPO = "https://github.com/OSegun/scoutfield.git"
BRANCH = "main"
SRC = "/kaggle/working/scoutfield"

# Cloned if absent, fast-forwarded if present. The second half is not optional:
# /kaggle/working persists between sessions when persistence is on, so a
# clone-if-missing alone pins the session to whatever commit was first cloned and
# every later push is silently ignored — the run then reports a stale commit while
# appearing to work. Local edits inside the clone are discarded, deliberately; the
# checkout is disposable and the repository is the record.
if not os.path.isdir(os.path.join(SRC, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, SRC],
                   check=True)
else:
    subprocess.run(["git", "-C", SRC, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", SRC, "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", SRC, "clean", "-fd"], check=True)
os.chdir(SRC)

# Printed, not assumed: `main` moves, so the commit is the only honest record of what
# this run actually executed. Quote it beside any number this notebook produces.
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

# Dependencies come from the CLONED file, so nothing here depends on
# raw.githubusercontent.com. Kaggle already ships torch, torchvision, numpy, pandas,
# scikit-learn and pillow — reinstalling them burns GPU quota and risks a CUDA
# mismatch against the driver on the machine, so they are not in that file.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                "requirements-kaggle.txt"], check=True)
# Editable, --no-deps: the line above already installed the pilot and the rest.
# `python experiments/01_....py` puts `experiments/` on sys.path, not the repo root,
# so `import scoutfield` needs a real install rather than the working directory.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
               check=True)

# Only /kaggle/working survives the session, and `utils.paths` writes there. The repo
# also carries results/, checkpoints/ and figures/ (each holding just a .gitkeep), so
# repo-relative paths and absolute ones would otherwise point at different places and
# the sweep would write its CSV somewhere its own summary step cannot find it.
for _d in ("results", "checkpoints", "figures"):
    _target = f"/kaggle/working/{_d}"
    os.makedirs(_target, exist_ok=True)
    if os.path.islink(_d):
        continue
    if os.path.isdir(_d):
        shutil.rmtree(_d)          # a fresh clone puts only .gitkeep in here
    os.symlink(_target, _d)

import scoutfield
from scoutfield.utils.paths import is_kaggle, output_dir
from scoutfield.utils.seeding import seed_everything

print("scoutfield", scoutfield.__version__, "@", COMMIT,
      "| pilot pinned at", scoutfield.PILOT_TAG,
      "| kaggle:", is_kaggle())

rng = seed_everything(0)

OUT = output_dir("results")
print("writing results to", OUT)

## Run


In [ ]:
!while JOB_BUDGET=60 python experiments/04_sweep.py; do :; done
!python experiments/05_tau_sensitivity.py --config configs/sweep.yaml
!python experiments/make_figures.py